-------------------BUILD DATASET---------------------

In [2]:
import os
import shutil

In [3]:
ORGIGIN_DATASET_PATH = '/mnt/data1tb/vinh/TemporalGAN/dataset/o2s'
TARGET_DATASET_PATH = '/mnt/data1tb/vinh/TemporalGAN/dataset/o2s_4'
DATASET_PATH = '/mnt/data1tb/vinh/s2_s1_lc_O2S_3'

In [ ]:
# Build dataset
count = 0 # Debug
for folder_set in os.listdir(ORGIGIN_DATASET_PATH):
    folder_set_path = os.path.join(ORGIGIN_DATASET_PATH, folder_set)
    if not os.path.isdir(folder_set_path):
        print(f'Not folder. Skip: {folder_set_path}')
        continue

    for image_folder in os.listdir(folder_set_path):
        image_folder_path = os.path.join(folder_set_path, image_folder)
        if not os.path.isdir(image_folder_path):
            print(f'Not folder. Skip: {image_folder_path}')
            continue
    
        for image_name in os.listdir(image_folder_path):
            if not image_name.endswith('.png'):
                print(f'Not png format. Skip: {image_name}')
                continue
            image_path = os.path.join(image_folder_path, image_name)
            target_path = image_path.replace('o2s', os.path.basename(TARGET_DATASET_PATH))
            os.makedirs(os.path.dirname(target_path), exist_ok=True)
            
            base = os.path.splitext(image_name)[0]

            if image_folder == 'lc_2048':
                source_name = f'{base}_colored.png'
            else:
                source_name = image_name

            source_path = os.path.join(DATASET_PATH, image_folder, source_name)
            if not os.path.exists(source_path):
                print(f'[MISS] {source_path}')
                continue
            
            shutil.copy(source_path, target_path)


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

ROOT_DIR = "/mnt/data1tb/vinh/TemporalGAN/infer-testset-o2s_4-results-v1_1"
OUT_DIR = "histograms"
os.makedirs(OUT_DIR, exist_ok=True)

for sample_name in sorted(os.listdir(ROOT_DIR)):
    sample_dir = os.path.join(ROOT_DIR, sample_name)
    if not os.path.isdir(sample_dir):
        continue

    input_path  = os.path.join(sample_dir, "s1.png")
    target_path = os.path.join(sample_dir, "s1_gen.png")

    if not (os.path.exists(input_path) and os.path.exists(target_path)):
        continue

    img_in  = cv2.imread(input_path,  cv2.IMREAD_GRAYSCALE)
    img_tgt = cv2.imread(target_path, cv2.IMREAD_GRAYSCALE)

    if img_in is None or img_tgt is None:
        continue

    p_in  = img_in.flatten()
    p_tgt = img_tgt.flatten()

    plt.figure(figsize=(5, 3))
    plt.hist(p_in,  bins=256, range=(0,255), density=True, alpha=0.6, label="Input")
    plt.hist(p_tgt, bins=256, range=(0,255), density=True, alpha=0.6, label="Target")
    plt.legend(fontsize=8)
    plt.xlabel("Pixel (0–255)")
    plt.ylabel("Density")
    plt.title(sample_name)
    plt.grid(alpha=0.3)
    plt.tight_layout()

    out_path = os.path.join(OUT_DIR, f"{sample_name}.png")
    plt.savefig(out_path, dpi=150)
    plt.close()



# REMOVE _COLORED IN NAME OF LANDCOVER

In [ ]:
folder_path = "/mnt/data1tb/vinh/data_tumlum/lc_5_256_340sgdm"
import os
count = 0
for file_name in os.listdir(folder_path):
    current_path = os.path.join(folder_path, file_name)
    new_path = current_path.replace('_colored', '')
    # Rename
    os.rename(current_path, new_path)
    count +=1
    print('DONE')

print(f'OK [{count}/{len(os.listdir(folder_path))}]') 

Split train/valid/test Dataset for TSGAN O2S <br/>
Đối với các tên mảnh C gốc, thì 80% số mảnh C gốc sẽ về train, với mỗi mảnh C gốc sẽ là toàn bộ các bản cắt. Còn lại về valid. Tương tự đối với tên mảnh D, E, F gốc.

In [ ]:
import os
import shutil
from tqdm import tqdm
from collections import defaultdict
import random

# ================= CẤU HÌNH (CONFIG) =================
SOURCE_DIR = '/mnt/data1tb/vinh/data_tumlum/TSGAN_data_collection/data_filtered_TSGAN'
# Đường dẫn folder đích để lưu dataset chuẩn sau khi chia
OUTPUT_DIR = '/mnt/data1tb/vinh/TemporalGAN/dataset/o2s_splited'

# Tỷ lệ Train / Valid / Test (Hãy đảm bảo tổng là 1.0)
TRAIN_RATIO = 0.8
VAL_RATIO = 0.15
TEST_RATIO = 0.05

# Hành động khi lấy dữ liệu: Chọn 'copy' (an toàn) hoặc 'move' (chuyển qua luôn, giảm dung lượng)
FILE_ACTION = 'move' # 'copy' hoặc 'move'

# Seed cố định để kết quả đảm bảo tính tái tạo giống hệt nhau ở mọi lần
RANDOM_SEED = 42
# =====================================================

def ensure_dir(d):
    if not os.path.exists(d):
        os.makedirs(d)

def parse_filename(patch_filename):
    """
    Input: '0_0_F_48_93_B_d_4.npy'
    Output: 
        - scene_id: 'F_48_93_B_d_4' (Tên ảnh gốc)
        - category: 'F' (Chữ cái đầu tiên để phân loại tính đa dạng dữ liệu)
    """
    name_no_ext = os.path.splitext(patch_filename)[0]
    parts = name_no_ext.split('_')
    
    # Scene name bắt đầu từ phần tử thứ 2 trở đi (bỏ row, col)
    scene_id = "_".join(parts[2:]) 
    category = parts[2] 
    
    return scene_id, category

def process_triplet(triplet_name):
    print(f"\n{'='*60}")
    print(f"BẮT ĐẦU XỬ LÝ TRIPLET: {triplet_name}")
    print(f"{'='*60}")
    
    # Khởi tạo seed riêng trong mỗi triplet để nếu bạn chỉ muốn chia 1 phần cũng ra được KQ y hệt
    random.seed(RANDOM_SEED)

    triplet_dir = os.path.join(SOURCE_DIR, triplet_name)
    sub_folders = ['s1', 's2', 'lc']
    
    # Kiểm tra đường dẫn có chứa đầy đủ subfolder ko
    for sub in sub_folders:
        if not os.path.exists(os.path.join(triplet_dir, sub)):
            print(f"Lỗi: Không tìm thấy folder {sub} trong {triplet_dir}")
            return

    # Lấy thư mục `s2` làm chuẩn để đếm file và phân vùng
    reference_dir = os.path.join(triplet_dir, 's2')
    ref_files = [f for f in os.listdir(reference_dir) if f.endswith('.npy')]
    
    # Gom nhóm patch theo Scene và Category
    category_to_scenes = defaultdict(list)
    scene_to_patches = defaultdict(list)
    seen_scenes = set()

    for f in tqdm(ref_files, desc=f"Phân tích file trong {triplet_name}"):
        scene_id, category = parse_filename(f)
        scene_to_patches[scene_id].append(f)
        
        if scene_id not in seen_scenes:
            category_to_scenes[category].append(scene_id)
            seen_scenes.add(scene_id)
            
    # STRATIFIED SPLIT
    train_scenes_final = []
    val_scenes_final = []
    test_scenes_final = []
    
    print(f"\n>>> Thống kê và Phân chia (Train={TRAIN_RATIO}/Val={VAL_RATIO}/Test={TEST_RATIO}):")
    print(f"{'Category':<10} | {'Total':<8} | {'Train':<8} | {'Val':<8} | {'Test':<8}")
    print("-" * 55)
    
    sorted_categories = sorted(category_to_scenes.keys())
    
    for cat in sorted_categories:
        scenes = category_to_scenes[cat]
        # Xáo trộn với ngẫu nhiên để chọn train/val/test
        random.shuffle(scenes)
        
        n_total = len(scenes)
        n_train = int(n_total * TRAIN_RATIO)
        n_val = int(n_total * VAL_RATIO)
        
        train_subset = scenes[:n_train]
        val_subset = scenes[n_train:n_train + n_val]
        # Nắm những thằng còn lại cho test
        test_subset = scenes[n_train + n_val:]
        
        train_scenes_final.extend(train_subset)
        val_scenes_final.extend(val_subset)
        test_scenes_final.extend(test_subset)
        
        print(f"{cat:<10} | {n_total:<8} | {len(train_subset):<8} | {len(val_subset):<8} | {len(test_subset):<8}")

    # DI CHUYỂN HOẶC SAO CHÉP FILE ĐỒNG BỘ CẢ 3 FOLDERS
    print(f"\n>>> Bắt đầu action '{FILE_ACTION}' file cho: {triplet_name}...")
    
    phases = ['train', 'valid', 'test']
    for phase in phases:
        for sub in sub_folders:
            ensure_dir(os.path.join(OUTPUT_DIR, phase, triplet_name, sub))

    def process_data(scene_list, phase):
        count_files = 0
        for scene in tqdm(scene_list, desc=f"{FILE_ACTION.capitalize()}ing to {phase.upper()}"):
            patches = scene_to_patches[scene]
            for patch_name in patches:
                for sub in sub_folders:
                    src = os.path.join(triplet_dir, sub, patch_name)
                    dst = os.path.join(OUTPUT_DIR, phase, triplet_name, sub, patch_name)
                    
                    if os.path.exists(src):
                        if FILE_ACTION == 'copy':
                            shutil.copy2(src, dst)
                        elif FILE_ACTION == 'move':
                            shutil.move(src, dst)
                    else:
                        print(f"Warning: Thiếu file {src}")
                count_files += 1
        return count_files

    n_train_files = process_data(train_scenes_final, 'train')
    n_val_files = process_data(val_scenes_final, 'valid')
    n_test_files = process_data(test_scenes_final, 'test')

    print(f"\n=> HOÀN TẤT CHO {triplet_name}!")
    print(f"Tổng số Scenes (Train/Val/Test): {len(train_scenes_final)} / {len(val_scenes_final)} / {len(test_scenes_final)}")
    print(f"Tổng số Patches (Train/Val/Test): {n_train_files} / {n_val_files} / {n_test_files}")

def main():
    triplets_to_process = ['triplet_vv', 'triplet_vh']
    
    for t in triplets_to_process:
        target_path = os.path.join(SOURCE_DIR, t)
        if os.path.exists(target_path):
            process_triplet(t)
        else:
            print(f"BỎ QUA {t}: Không tìm thấy thư mục {target_path}")
            
    print(f"\n[DONE] Toàn bộ dữ liệu đã được phân chia lưu tại: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()


Visualize triplets (sau split) để xác thực tính đồng bộ

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

# ================= CẤU HÌNH (CONFIG) =================
# Thư mục đã phân chia xong
SPLIT_DIR = '/mnt/data1tb/vinh/TemporalGAN/dataset/o2s_splited'

# Giới hạn số lượng triplet ngẫu nhiên muốn visualize cho MỖI THƯ MỤC
LIMIT_CHECK = 3  

# =====================================================

def plot_triplet(s1_path, s2_path, lc_path, info_title):
    try:
        s1 = np.load(s1_path)
        s2 = np.load(s2_path)
        lc = np.load(lc_path)
    except Exception as e:
        print(f"Lỗi khi load numpy array: {e}")
        return

    # Khởi tạo frame hiển thị matplotlib
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(info_title, fontsize=14, fontweight='bold')

    # Xử lý hiển thị S1 (Sentinel-1)
    # S1 thường lấy channel 0 hoặc tính amplitude. Ở đây hiển thị dưới dạng Grayscale
    if len(s1.shape) == 3:
        s1_view = s1[:, :, 0] # Lấy channel đầu
    else:
        s1_view = s1
    
    # Chuẩn hoá S1 để view rõ hơn
    s1_min, s1_max = np.percentile(s1_view, (2, 98))
    s1_view = np.clip(s1_view, s1_min, s1_max)
    if s1_max != s1_min:
         s1_view = (s1_view - s1_min) / (s1_max - s1_min)

    axes[0].imshow(s1_view, cmap='gray')
    axes[0].set_title(f"S1 Patch\nShape: {s1.shape}")
    axes[0].axis('off')

    # Xử lý hiển thị S2 (Sentinel-2)
    # S2 thường là ảnh n kênh, lấy 3 channel chuẩn RGB hoặc tuỳ mapping
    if len(s2.shape) == 3 and s2.shape[2] >= 3:
        # Lấy 3 vùng giá trị đầu (nếu s2 xếp RGB đầu tiên)
        s2_view = s2[:, :, :3] 
    else:
        s2_view = s2

    # Chuẩn hoá S2 để view rõ hơn
    s2_min, s2_max = np.percentile(s2_view, (2, 98))
    s2_view = np.clip(s2_view, s2_min, s2_max)
    if s2_max != s2_min:
          s2_view = (s2_view - s2_min) / (s2_max - s2_min)

    axes[1].imshow(s2_view)
    axes[1].set_title(f"S2 Patch\nShape: {s2.shape}")
    axes[1].axis('off')

    # Xử lý hiển thị LC (Landcover)
    if len(lc.shape) == 3:
         # Landcover map có thể có 3 channel RGB, hoặc 1 channel Segmentation Mask
         lc_view = lc[:, :, 0] if lc.shape[2] not in [3,4] else lc
    else:
         lc_view = lc
         
    axes[2].imshow(lc_view, cmap='jet' if len(lc_view.shape)==2 else None)
    axes[2].set_title(f"Land Cover Patch\nShape: {lc.shape}")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

def verify_split(split_phase, triplet_type):
    print(f"\n[{split_phase.upper()} | {triplet_type.upper()}] Kiểm tra ngẫu nhiên {LIMIT_CHECK} samples:")
    base_dir = os.path.join(SPLIT_DIR, split_phase, triplet_type)
    s2_dir = os.path.join(base_dir, 's2')
    
    if not os.path.exists(s2_dir):
        print(f"Bỏ qua (Không tìm thấy thư mục: {s2_dir})")
        return
        
    s2_files = [f for f in os.listdir(s2_dir) if f.endswith('.npy')]
    
    if not s2_files:
        print(f"Thư mục trống: {s2_dir}")
        return
        
    # Lựa chọn ngẫu nhiên các mẫu để verify
    num_samples = min(LIMIT_CHECK, len(s2_files))
    sample_files = random.sample(s2_files, num_samples)
    
    for filename in sample_files:
        s1_path = os.path.join(base_dir, 's1', filename)
        s2_path = os.path.join(base_dir, 's2', filename)
        lc_path = os.path.join(base_dir, 'lc', filename)
        
        info_title = f"Phase: {split_phase.upper()} | Triplet: {triplet_type.upper()} | File: {filename}"
        
        if os.path.exists(s1_path) and os.path.exists(s2_path) and os.path.exists(lc_path):
            plot_triplet(s1_path, s2_path, lc_path, info_title)
        else:
            print(f"MISSING DATA for file: {filename}")

def main_visualizer():
    phases = ['train', 'valid', 'test']
    triplets = ['triplet_vv', 'triplet_vh']
    
    for p in phases:
        for t in triplets:
             verify_split(p, t)

# Bắt đầu chạy visualize
main_visualizer()
